In [8]:
"""
Global Superstore - Phase 2 | Notebook 03
RFM Customer Segmentation
File: 02_Python/03_RFM_Segmentation.py

Chay: python 02_Python/03_RFM_Segmentation.py
Output: data/rfm_segments.csv (dung cho Power BI)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings("ignore")

# ── Setup ─────────────────────────────────────────────────────────────────────
DB_FILE   = r"C:\DA\global-superstore-analysis\data\superstore.db"
CHART_DIR = r"C:\DA\global-superstore-analysis\02_Python\charts"
DATA_DIR  = r"C:\DA\global-superstore-analysis\data"
os.makedirs(CHART_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})

COLORS = {
    "Champions":          "#1F4E79",
    "Loyal Customers":    "#2E75B6",
    "Potential Loyalists":"#70AD47",
    "At Risk":            "#FFC000",
    "Cant Lose Them":     "#ED7D31",
    "Hibernating":        "#FF0000",
    "New Customers":      "#9E480E",
    "Lost":               "#595959",
}

print("=" * 60)
print("GLOBAL SUPERSTORE - RFM SEGMENTATION")
print("=" * 60)

# ── Load data ──────────────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_FILE)
df   = pd.read_sql("SELECT * FROM orders", conn)
conn.close()

df["order_date"] = pd.to_datetime(df["order_date"])
print(f"\nData loaded: {df.shape[0]:,} rows")

# ── BUOC 1: Tinh R, F, M cho tung khach hang ──────────────────────────────────
print("\n[1/5] Tinh R, F, M values...")

snapshot_date = df["order_date"].max() + pd.Timedelta(days=1)
print(f"      Snapshot date: {snapshot_date.date()}")

rfm = df.groupby("customer_id").agg(
    customer_name = ("customer_name", "first"),
    segment       = ("segment",       "first"),
    recency       = ("order_date",    lambda x: (snapshot_date - x.max()).days),
    frequency     = ("order_id",      "nunique"),
    monetary      = ("sales",         "sum"),
).reset_index()

print(f"      Total customers: {len(rfm):,}")
print(f"\n      Recency  — min: {rfm['recency'].min()} days | "
      f"max: {rfm['recency'].max()} days | "
      f"mean: {rfm['recency'].mean():.0f} days")
print(f"      Frequency — min: {rfm['frequency'].min()} | "
      f"max: {rfm['frequency'].max()} | "
      f"mean: {rfm['frequency'].mean():.1f}")
print(f"      Monetary  — min: ${rfm['monetary'].min():.0f} | "
      f"max: ${rfm['monetary'].max():,.0f} | "
      f"mean: ${rfm['monetary'].mean():.0f}")

# ── BUOC 2: Tinh R, F, M scores (1-4) ────────────────────────────────────────
print("\n[2/5] Cham diem R, F, M (thang 1-4)...")

# R: thap = tot (mua gan day) -> dao nguoc
rfm["R_score"] = pd.qcut(rfm["recency"],  q=4, labels=[4, 3, 2, 1])
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"),
                          q=4, labels=[1, 2, 3, 4])
rfm["M_score"] = pd.qcut(rfm["monetary"], q=4, labels=[1, 2, 3, 4])

rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)
rfm["RFM_Score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

print(f"      Score range: {rfm['RFM_Score'].min()} - {rfm['RFM_Score'].max()}")

# ── BUOC 3: Phan loai Segment ─────────────────────────────────────────────────
print("\n[3/5] Phan loai khach hang thanh 8 nhom...")

def rfm_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    score   = row["RFM_Score"]
    if r >= 4 and f >= 3 and m >= 3:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 3 and f <= 2:
        return "Potential Loyalists"
    elif r == 2 and f >= 2 and m >= 2:
        return "At Risk"
    elif r <= 2 and f >= 3 and m >= 3:
        return "Cant Lose Them"
    elif r >= 3 and f == 1:
        return "New Customers"
    elif r == 2 and score <= 5:
        return "Hibernating"
    else:
        return "Lost"

rfm["RFM_Segment"] = rfm.apply(rfm_segment, axis=1)

# Segment summary
seg_summary = rfm.groupby("RFM_Segment").agg(
    customer_count = ("customer_id", "count"),
    avg_recency    = ("recency",     "mean"),
    avg_frequency  = ("frequency",  "mean"),
    avg_monetary   = ("monetary",   "mean"),
    total_monetary = ("monetary",   "sum"),
).reset_index()
seg_summary["revenue_pct"] = (
    seg_summary["total_monetary"] / seg_summary["total_monetary"].sum() * 100
)
seg_summary = seg_summary.sort_values("total_monetary", ascending=False)

print("\n      Segment Distribution:")
print(seg_summary[["RFM_Segment","customer_count","avg_monetary",
                    "total_monetary","revenue_pct"]].to_string(index=False))

# ── BUOC 4: Tao Visualizations ────────────────────────────────────────────────

# CHART 11: Segment Distribution (Count + Revenue)
print("\n[Chart 11] Segment Distribution...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart - Customer count
seg_sorted = seg_summary.sort_values("customer_count", ascending=False)
colors_pie = [COLORS.get(s, "#AAAAAA") for s in seg_sorted["RFM_Segment"]]
wedges, texts, autotexts = ax1.pie(
    seg_sorted["customer_count"],
    labels=seg_sorted["RFM_Segment"],
    colors=colors_pie,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.75,
)
for t in autotexts:
    t.set_fontsize(9)
ax1.set_title("Customer Distribution\nby Segment", fontweight="bold")

# Bar chart - Revenue contribution
seg_rev = seg_summary.sort_values("total_monetary", ascending=True)
bar_colors = [COLORS.get(s, "#AAAAAA") for s in seg_rev["RFM_Segment"]]
bars = ax2.barh(seg_rev["RFM_Segment"], seg_rev["total_monetary"]/1e6,
                color=bar_colors, height=0.6)
for bar, pct, cnt in zip(bars,
                          seg_rev["revenue_pct"],
                          seg_rev["customer_count"]):
    ax2.text(bar.get_width() + 0.02,
             bar.get_y() + bar.get_height()/2,
             f"${bar.get_width():.1f}M  ({pct:.1f}%)  n={cnt}",
             va="center", fontsize=9)
ax2.set_xlabel("Total Revenue (Million USD)")
ax2.set_title("Revenue Contribution\nby Segment", fontweight="bold")
ax2.set_xlim(0, seg_rev["total_monetary"].max()/1e6 * 1.4)

plt.suptitle("Chart 11 — RFM Customer Segmentation",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_11_rfm_segments.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_11_rfm_segments.png")

# CHART 12: RFM Score Heatmap
print("[Chart 12] RFM Heatmap...")

heatmap_data = rfm.groupby(["R_score", "F_score"])["monetary"].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".0f",
    cmap="YlOrRd",
    ax=ax,
    cbar_kws={"label": "Avg Monetary (USD)"},
    linewidths=0.5,
)
ax.set_xlabel("Frequency Score")
ax.set_ylabel("Recency Score (4=Most Recent)")
ax.set_title("Chart 12 — RFM Heatmap\n"
             "Avg Monetary Value by Recency x Frequency Score",
             fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_12_rfm_heatmap.png"), dpi=150)
plt.close()
print("   Saved: chart_12_rfm_heatmap.png")

# CHART 13: Scatter R vs M colored by Segment
print("[Chart 13] RFM Scatter Plot...")

sample = rfm.sample(min(len(rfm), len(rfm)), random_state=42)
fig, ax = plt.subplots(figsize=(10, 6))

for seg, grp in sample.groupby("RFM_Segment"):
    ax.scatter(grp["recency"], grp["monetary"],
               c=COLORS.get(seg, "#AAAAAA"),
               label=seg, alpha=0.7, s=40,
               edgecolors="white", linewidth=0.3)

ax.set_xlabel("Recency (days since last purchase) — Lower = Better")
ax.set_ylabel("Monetary (Total Sales USD)")
ax.set_title("Chart 13 — Customer Map: Recency vs Monetary by Segment",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left",
          fontsize=9, framealpha=0.0)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_13_rfm_scatter.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_13_rfm_scatter.png")

# CHART 14: Avg Metrics by Segment (Radar-style bar)
print("[Chart 14] Segment Profile...")

metrics = ["avg_recency", "avg_frequency", "avg_monetary"]
seg_profile = seg_summary.set_index("RFM_Segment")[metrics].copy()

# Normalize 0-1 cho de so sanh
seg_profile_norm = seg_profile.copy()
seg_profile_norm["avg_recency"]   = 1 - (seg_profile["avg_recency"]   / seg_profile["avg_recency"].max())
seg_profile_norm["avg_frequency"] = seg_profile["avg_frequency"] / seg_profile["avg_frequency"].max()
seg_profile_norm["avg_monetary"]  = seg_profile["avg_monetary"]  / seg_profile["avg_monetary"].max()
seg_profile_norm = seg_profile_norm.sort_values("avg_monetary", ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(seg_profile_norm))
w = 0.25

bars1 = ax.bar(x - w,   seg_profile_norm["avg_recency"],   w, label="Recency (inv)",  color="#2E75B6", alpha=0.85)
bars2 = ax.bar(x,       seg_profile_norm["avg_frequency"], w, label="Frequency",       color="#70AD47", alpha=0.85)
bars3 = ax.bar(x + w,   seg_profile_norm["avg_monetary"],  w, label="Monetary",        color="#FFC000", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(seg_profile_norm.index, rotation=20, ha="right")
ax.set_ylabel("Normalized Score (0-1)")
ax.set_title("Chart 14 — Segment Profile: R, F, M Normalized Scores",
             fontsize=13, fontweight="bold", pad=12)
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_14_segment_profile.png"), dpi=150)
plt.close()
print("   Saved: chart_14_segment_profile.png")

# ── BUOC 5: Export CSV cho Power BI ───────────────────────────────────────────
print("\n[5/5] Export rfm_segments.csv cho Power BI...")

export_cols = [
    "customer_id", "customer_name", "segment",
    "recency", "frequency", "monetary",
    "R_score", "F_score", "M_score", "RFM_Score", "RFM_Segment"
]
rfm_export = rfm[export_cols].copy()
rfm_export["monetary"] = rfm_export["monetary"].round(2)

csv_path = os.path.join(DATA_DIR, "rfm_segments.csv")
rfm_export.to_csv(csv_path, index=False)
print(f"      Saved: {csv_path}")
print(f"      Rows  : {len(rfm_export):,}")
print(f"      Cols  : {list(rfm_export.columns)}")

# ── KEY INSIGHTS ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("KEY INSIGHTS - RFM SEGMENTATION")
print("=" * 60)

champ = rfm[rfm["RFM_Segment"] == "Champions"]
loyal = rfm[rfm["RFM_Segment"] == "Loyal Customers"]
risk  = rfm[rfm["RFM_Segment"] == "At Risk"]
lost  = rfm[rfm["RFM_Segment"] == "Lost"]
total_rev = rfm["monetary"].sum()

print(f"""
Total Customers : {len(rfm):,}

Champions ({len(champ)} customers = {len(champ)/len(rfm)*100:.1f}%):
  Revenue      : ${champ['monetary'].sum():,.0f}
  Revenue share: {champ['monetary'].sum()/total_rev*100:.1f}% of total
  Avg spending : ${champ['monetary'].mean():,.0f}
  Avg frequency: {champ['frequency'].mean():.1f} orders
  Avg recency  : {champ['recency'].mean():.0f} days

Loyal Customers ({len(loyal)} customers):
  Avg spending : ${loyal['monetary'].mean():,.0f}
  Champions spend {champ['monetary'].mean()/loyal['monetary'].mean():.1f}x more than Loyal

At Risk ({len(risk)} customers):
  Avg spending : ${risk['monetary'].mean():,.0f}
  Action needed: Win-back campaign immediately

Lost ({len(lost)} customers):
  Revenue lost : ${lost['monetary'].sum():,.0f}
  Low priority: High cost to reactivate

Pareto Check:
  Top 20% customers generate {rfm.nlargest(int(len(rfm)*0.2), 'monetary')['monetary'].sum()/total_rev*100:.1f}% of revenue

Strategy:
  Champions      -> VIP rewards, early access, personal thank-you
  Loyal          -> Loyalty program invitation
  At Risk        -> Win-back email + 15% discount voucher
  New Customers  -> Onboarding sequence, 2nd purchase incentive
  Lost           -> Last-chance offer, then remove from list
""")

print(f" Charts saved : {CHART_DIR}")
print(f" CSV exported : {csv_path}")
print("\n" + "=" * 60)
print("HOAN THANH Notebook 03 - RFM Segmentation!")
print("Buoc tiep theo: 02_Python/04_Geo_Analysis.py")
print("=" * 60)

GLOBAL SUPERSTORE - RFM SEGMENTATION

Data loaded: 51,290 rows

[1/5] Tinh R, F, M values...
      Snapshot date: 2015-01-01
      Total customers: 1,590

      Recency  — min: 1 days | max: 1207 days | mean: 85 days
      Frequency — min: 1 | max: 41 | mean: 16.2
      Monetary  — min: $7 | max: $35,668 | mean: $7951

[2/5] Cham diem R, F, M (thang 1-4)...
      Score range: 3 - 12

[3/5] Phan loai khach hang thanh 8 nhom...

      Segment Distribution:
        RFM_Segment  customer_count  avg_monetary  total_monetary  revenue_pct
          Champions             294  14293.032167    4.202151e+06    33.238290
    Loyal Customers             273  13945.112159    3.807016e+06    30.112834
            At Risk             280  10679.527391    2.990268e+06    23.652499
               Lost             378   1994.384839    7.538775e+05     5.963040
Potential Loyalists             232   2376.649728    5.513827e+05     4.361342
     Cant Lose Them              18  11404.658212    2.052838e+05  